# T5 (Text-To-Text Transfer Transformer): Deep Mathematical Description

T5 introduced several innovations in natural language processing:

1. **Unified text-to-text framework:**
   T5 reformulated all NLP tasks into a text-to-text format:

   $f(x) = y$

   where $x$ is the input text (possibly including a task description), and $y$ is the target output text.

2. **Transfer learning:**
   T5 uses a pre-training and fine-tuning approach. The pre-training objective is a denoising task:

   $L_{pretrain} = -\log P(x | c(x))$

   where $x$ is the original text, and $c(x)$ is a corrupted version of $x$.

3. **Scaled dot-product attention:**
   The core of the Transformer architecture used in T5 is the attention mechanism:

   $Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

   where $Q$, $K$, and $V$ are query, key, and value matrices, and $d_k$ is the dimension of the key vectors.

4. **Position-wise feed-forward networks:**
   Each layer in T5 includes a feed-forward network:

   $FFN(x) = \max(0, xW_1 + b_1)W_2 + b_2$

   where $W_1$, $W_2$, $b_1$, and $b_2$ are learnable parameters.

5. **Layer normalization:**
   T5 uses layer normalization:

   $LN(x) = \alpha * \frac{x - \mu}{\sigma + \epsilon} + \beta$

   where $\mu$ and $\sigma$ are the mean and standard deviation of the inputs, $\alpha$ and $\beta$ are learnable parameters, and $\epsilon$ is a small constant for numerical stability.

## References

1. [Raffel et al. (2020). "Exploring the limits of transfer learning with a unified text-to-text transformer."](https://jmlr.org/papers/v21/20-074.html)

2. [Vaswani et al. (2017). "Attention is all you need."](https://arxiv.org/abs/1706.03762)

3. [Ba et al. (2016). "Layer normalization."](https://arxiv.org/abs/1607.06450)

4. [Devlin et al. (2018). "BERT: Pre-training of deep bidirectional transformers for language understanding."](https://arxiv.org/abs/1810.04805)

In [1]:
!pip install numpy tensorflow==2.13.0 keras scikit-learn matplotlib

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
import random
import string

class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k):
        super().__init__()
        self.d_k = d_k

    def forward(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = F.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention(self.d_k)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attn_output = self.attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(attn_output)

class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

class T5EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)

    def forward(self, x, mask=None):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + attn_output)
        ff_output = self.feed_forward(x)
        x = self.norm2(x + ff_output)
        return x

class T5DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.norm3 = LayerNorm(d_model)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + attn_output)
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + attn_output)
        ff_output = self.feed_forward(x)
        x = self.norm3(x + ff_output)
        return x

class T5Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([T5EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])

    def forward(self, x, mask=None):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x, mask)
        return x

class T5Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([T5DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return self.fc_out(x)

class T5(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.encoder = T5Encoder(vocab_size, d_model, num_heads, d_ff, num_layers)
        self.decoder = T5Decoder(vocab_size, d_model, num_heads, d_ff, num_layers)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        return dec_output

    def encode(self, src, src_mask=None):
        return self.encoder(src, src_mask)

    def decode(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        return self.decoder(tgt, enc_output, src_mask, tgt_mask)

class SyntheticDataset(Dataset):
    def __init__(self, num_samples, max_length):
        self.data = []
        for _ in range(num_samples):
            length = random.randint(3, max_length)
            sentence = ' '.join(''.join(random.choices(string.ascii_lowercase, k=random.randint(1, 5)))
                                for _ in range(length))
            self.data.append(sentence)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

class SimpleTokenizer:
    def __init__(self):
        self.vocab = list(string.ascii_lowercase) + [' ', '<start>', '<end>', '<pad>']
        self.token_to_id = {token: i for i, token in enumerate(self.vocab)}
        self.id_to_token = {i: token for token, i in self.token_to_id.items()}
        self.vocab_size = len(self.vocab)

    def encode(self, text, max_length):
        tokens = ['<start>'] + list(text) + ['<end>']
        ids = [self.token_to_id[token] for token in tokens]
        if len(ids) < max_length:
            ids += [self.token_to_id['<pad>']] * (max_length - len(ids))
        else:
            ids = ids[:max_length-1] + [self.token_to_id['<end>']]
        return torch.tensor(ids)

    def decode(self, ids):
        tokens = [self.id_to_token[id.item()] for id in ids]
        return ''.join(token for token in tokens if token not in ['<start>', '<end>', '<pad>'])

def train(model, dataloader, tokenizer, device, num_epochs=10):
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id['<pad>'])
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in dataloader:
            optimizer.zero_grad()

            src = tokenizer.encode(batch[0], max_length=20).unsqueeze(0).to(device)
            tgt = tokenizer.encode(' '.join(batch[0].split()[::-1]), max_length=20).unsqueeze(0).to(device)

            src_mask = (src != tokenizer.token_to_id['<pad>']).unsqueeze(1).unsqueeze(2)
            tgt_mask = (tgt != tokenizer.token_to_id['<pad>']).unsqueeze(1).unsqueeze(2)
            tgt_mask = tgt_mask & torch.tril(torch.ones((1, tgt.size(1), tgt.size(1)), device=device)).bool()

            output = model(src, tgt[:, :-1], src_mask, tgt_mask[:, :, :-1, :-1])
            loss = criterion(output.contiguous().view(-1, tokenizer.vocab_size), tgt[:, 1:].contiguous().view(-1))

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}")

def inference(model, tokenizer, device, input_text):
    model.eval()
    src = tokenizer.encode(input_text, max_length=20).unsqueeze(0).to(device)
    src_mask = (src != tokenizer.token_to_id['<pad>']).unsqueeze(1).unsqueeze(2)

    enc_output = model.encode(src, src_mask)

    tgt = torch.tensor([[tokenizer.token_to_id['<start>']]], device=device)

    for _ in range(20):  # Max length of 20
        tgt_mask = torch.tril(torch.ones((1, tgt.size(1), tgt.size(1)), device=device)).bool()
        out = model.decode(tgt, enc_output, src_mask, tgt_mask)
        prob = out[:, -1]
        _, next_word = torch.max(prob, dim=1)
        tgt = torch.cat([tgt, next_word.unsqueeze(0)], dim=1)

        if next_word == tokenizer.token_to_id['<end>']:
            break

    return tokenizer.decode(tgt[0])

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Create dataset and dataloader
    dataset = SyntheticDataset(num_samples=1000, max_length=10)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

    # Initialize tokenizer
    tokenizer = SimpleTokenizer()

    # Initialize model
    model = T5(tokenizer.vocab_size, d_model=64, num_heads=4, d_ff=256, num_layers=3).to(device)

    # Train the model
    train(model, dataloader, tokenizer, device, num_epochs=20)

    # Test the model
    test_sentences = [
        "hello world",
        "machine learning is fun",
        "pytorch is great"
    ]

    for sentence in test_sentences:
        output = inference(model, tokenizer, device, sentence)
        print(f"Input: {sentence}")
        print(f"Output: {output}")
        print(f"Expected: {' '.join(sentence.split()[::-1])}")
        print()

Epoch 1/20, Loss: 3.0572
Epoch 2/20, Loss: 2.9000
Epoch 3/20, Loss: 2.8134
Epoch 4/20, Loss: 2.7534
Epoch 5/20, Loss: 2.7075
Epoch 6/20, Loss: 2.6649
Epoch 7/20, Loss: 2.6319
Epoch 8/20, Loss: 2.6074
Epoch 9/20, Loss: 2.5779
Epoch 10/20, Loss: 2.5584
Epoch 11/20, Loss: 2.5372
Epoch 12/20, Loss: 2.5128
Epoch 13/20, Loss: 2.4942
Epoch 14/20, Loss: 2.4761
Epoch 15/20, Loss: 2.4522
Epoch 16/20, Loss: 2.4372
Epoch 17/20, Loss: 2.4104
Epoch 18/20, Loss: 2.3939
Epoch 19/20, Loss: 2.3708
Epoch 20/20, Loss: 2.3535
Input: hello world
Output: ldo lwerhr l
Expected: world hello

Input: machine learning is fun
Output: nn hian ennn hann
Expected: fun is learning machine

Input: pytorch is great
Output: tr ep ocrhr oya
Expected: great is pytorch



# Is this a T5 model?

This implementation is a simplified version inspired by the T5 (Text-To-Text Transfer Transformer) architecture, but it's not a full or exact implementation of the original T5 model as described in the "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer" paper.

## Key Differences

1. **Scale**: This is a much smaller model than the original T5, which had versions ranging from 60 million to 11 billion parameters.

2. **Pre-training**: The original T5 was pre-trained on a large corpus of text data. This implementation is trained from scratch on a small synthetic dataset.

3. **Objective**: The original T5 was designed for various text-to-text tasks. This implementation is specifically set up for a word reversal task as a simple demonstration.

4. **Architecture details**: While this implementation uses some ideas from T5 like the encoder-decoder structure, it doesn't include all the specific architectural choices of T5 (like the exact layer normalization placement).

5. **Tokenization**: T5 used SentencePiece tokenization, while this uses a very simple character-level tokenizer.

6. **No relative position embeddings**: The original T5 used relative position embeddings, which are not implemented here.

## Conclusion

This is not a full T5 implementation. It's a simplified transformer model inspired by some aspects of T5, designed as a learning exercise or starting point for understanding transformer architectures.

For practical applications requiring the actual T5 model, it's recommended to use implementations from libraries like Hugging Face's Transformers, which provide pre-trained models and more faithful implementations of the original architecture.